## 필수 · 기본 문제 1. 동적 padding과 PAD-mask 일치

### 문제 배경

Dataset에 저장할 때 모든 문장을 최대 길이로 채우면 짧은 batch에도 불필요한 PAD가 많습니다. 먼저 가변 길이로 tokenize하고 collator가 현재 batch의 최장 길이에만 맞추게 합니다.

### 시작 코드

```python
texts = ["금리 전망", "대표팀이 연장전 끝에 결승에 진출했다", "새 반도체 공개"]

def make_dynamic_batch(texts, max_length=16):
    raise NotImplementedError
```

### 수행 요구사항

1. Tokenizer 호출에서 `padding=False`, `truncation=True`를 사용하세요.
2. Encoding별 원래 길이를 기록하세요.
3. `DataCollatorWithPadding(return_tensors="pt")`으로 batch를 만드세요.
4. Batch 두 번째 축이 `min(max(original_lengths), max_length)`인지 확인하세요.
5. PAD ID 위치와 mask 0 위치의 완전 일치를 확인하세요.

### 제출 결과

- 원래 길이 list와 batch shape
- 문장별 PAD 수
- `mask_pad_consistent=True`
- `기본 문제 1 자동 검증: PASS`

### 자동 검증

```python
report = make_dynamic_batch(texts)
assert report["batch_shape"][0] == len(texts)
assert report["batch_shape"][1] == max(report["original_lengths"])
assert report["mask_pad_consistent"] is True
print("기본 문제 1 자동 검증: PASS")
```
    
    **자주 하는 실수**
    
    - Tokenizer에서 이미 `padding="max_length"`를 적용해 동적 padding 효과를 없앱니다.
    - Python list를 `torch.stack`해 길이 불일치 오류를 만듭니다.
    - PAD 수와 실제 token 수를 반대로 해석합니다.

접근 순서 · 입력 list의 최대 길이를 먼저 구한 뒤 PAD ID와 mask 0을 같은 개수만큼 붙입니다. 원본 길이만큼은 mask 1이어야 하므로 두 결과를 함께 만들면 불일치를 줄일 수 있습니다.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [3]:
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
texts = ["금리 전망", "대표팀이 연장전 끝에 결승에 진출했다", "새 반도체 공개"]

def make_dynamic_batch(texts, max_length=16):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    # 하나씩 tokenize하여 서로 다른 길이의 feature dictionary를 보존합니다.
    features = [tokenizer(text, padding=False, truncation=True,
                          max_length=max_length) for text in texts]
    original_lengths = [len(feature["input_ids"]) for feature in features]
    # Collator가 호출되는 순간 이 batch의 최장 길이까지만 PAD를 붙입니다.
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    batch = collator(features)
    pad_positions = batch["input_ids"].eq(tokenizer.pad_token_id)
    zero_mask_positions = batch["attention_mask"].eq(0)
    return {
        "batch_shape": tuple(batch["input_ids"].shape),
        "original_lengths": original_lengths,
        "pad_counts": pad_positions.sum(dim=1).tolist(),
        "mask_pad_consistent": torch.equal(pad_positions, zero_mask_positions),
    }

report = make_dynamic_batch(texts)
print(report)
assert report["batch_shape"][0] == len(texts)
assert report["batch_shape"][1] == max(report["original_lengths"])
assert report["mask_pad_consistent"] is True
print("기본 문제 1 자동 검증: PASS")

{'batch_shape': (3, 13), 'original_lengths': [4, 13, 5], 'pad_counts': [9, 0, 8], 'mask_pad_consistent': True}
기본 문제 1 자동 검증: PASS


상세 해설 · Collator는 이 세 문장 중 가장 긴 encoding까지만 PAD합니다. 두 번째 축의 정확한 숫자는 tokenizer 버전과 입력에 따라 달라질 수 있으므로 관계식과 mask 계약을 검증합니다

## 필수 · 기본 문제 2. DatasetDict `map` 전처리 파이프라인

### 문제 배경

Train·validation·test에 서로 다른 tokenization 옵션을 적용하면 평가 계약이 깨집니다. 하나의 batch 함수와 하나의 config로 세 split을 전처리하고 실제 batch까지 만듭니다.

### 시작 코드

```python
raw = {
    "train": {"id": ["tr0", "tr1", "tr2"], "text": ["금리 인상 전망", "대표팀 결승 진출", "AI 칩 공개"], "label": [0, 1, 2]},
    "validation": {"id": ["va0", "va1"], "text": ["수출 증가", "신인 선수 첫 승"], "label": [0, 1]},
    "test": {"id": ["te0", "te1"], "text": ["증시 반등", "보안 패치 배포"], "label": [0, 2]},
}

def build_dataset_pipeline(raw, max_length=16):
    raise NotImplementedError
```

### 수행 요구사항

1. `Dataset.from_dict`와 `DatasetDict`로 세 split을 만드세요.
2. `map(batched=True)` 함수에서 `padding=False`, truncation과 `max_length`를 적용하세요.
3. `labels`를 정수 list로 추가하되 원본 `id/text`는 오류 분석용으로 보존하세요.
4. Train 전체를 collator에 전달해 실제 batch를 만들고 shape·dtype을 반환하세요.
5. Split별 ID가 겹치지 않는지 자동 검사하세요.

### 제출 결과

- Split별 행 수·column names
- Train batch input/mask/labels shape·dtype
- Split ID disjoint와 PAD-mask PASS
- `기본 문제 2 자동 검증: PASS`

### 자동 검증

```python
report = build_dataset_pipeline(raw)
assert report["split_sizes"] == {"train": 3, "validation": 2, "test": 2}
assert report["id_disjoint"] and report["mask_pad_consistent"]
assert report["batch_shapes"]["labels"] == (3,)
print("기본 문제 2 자동 검증: PASS")
    ```
    **자주 하는 실수**
    
    - 문자열 `text`를 collator에 넘겨 Tensor 변환 오류를 냅니다.
    - `labels` 길이가 batch 행 수와 다릅니다.
    - 원본 ID를 제거해 split 중복과 오류 사례를 추적하지 못합니다.

파이프라인 핵심 · Tokenization 함수는 한 행이 아니라 열별 list 묶음을 받고, 반환 list 길이를 입력 행 수와 맞춥니다. Split별로 같은 함수를 적용한 뒤 collator가 batch 시점의 padding을 담당하게 합니다.

In [5]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
raw = {
    "train": {"id": ["tr0", "tr1", "tr2"], "text": ["금리 인상 전망", "대표팀 결승 진출", "AI 칩 공개"], "label": [0, 1, 2]},
    "validation": {"id": ["va0", "va1"], "text": ["수출 증가", "신인 선수 첫 승"], "label": [0, 1]},
    "test": {"id": ["te0", "te1"], "text": ["증시 반등", "보안 패치 배포"], "label": [0, 2]},
}

def build_dataset_pipeline(raw, max_length=16):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    # 먼저 split 이름과 원본 ID를 가진 DatasetDict를 고정합니다.
    dataset = DatasetDict({name: Dataset.from_dict(columns) for name, columns in raw.items()})

    def tokenize_batch(batch):
        encoded = tokenizer(batch["text"], padding=False, truncation=True,
                            max_length=max_length)
        encoded["labels"] = [int(label) for label in batch["label"]]
        return encoded

    # 세 split에 같은 함수와 max_length 계약을 적용합니다.
    tokenized = dataset.map(tokenize_batch, batched=True)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    # 문자열 열은 collator에 넘기지 않고 모델 입력 열만 선택합니다.
    feature_keys = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in tokenized["train"].column_names:
        feature_keys.append("token_type_ids")
    features = [{key: tokenized["train"][i][key] for key in feature_keys}
                for i in range(len(tokenized["train"]))]
    batch = collator(features)

    id_sets = {name: set(split["id"]) for name, split in dataset.items()}
    id_disjoint = not ((id_sets["train"] & id_sets["validation"]) |
                       (id_sets["train"] & id_sets["test"]) |
                       (id_sets["validation"] & id_sets["test"]))
    pad = batch["input_ids"].eq(tokenizer.pad_token_id)
    zero_mask = batch["attention_mask"].eq(0)
    return {
        "split_sizes": {name: len(split) for name, split in tokenized.items()},
        "columns": {name: split.column_names for name, split in tokenized.items()},
        "batch_shapes": {name: tuple(value.shape) for name, value in batch.items()},
        "batch_dtypes": {name: str(value.dtype) for name, value in batch.items()},
        "id_disjoint": id_disjoint,
        "mask_pad_consistent": torch.equal(pad, zero_mask),
    }

report = build_dataset_pipeline(raw)
print(report)
assert report["split_sizes"] == {"train": 3, "validation": 2, "test": 2}
assert report["id_disjoint"] and report["mask_pad_consistent"]
assert report["batch_shapes"]["labels"] == (3,)
print("기본 문제 2 자동 검증: PASS")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

{'split_sizes': {'train': 3, 'validation': 2, 'test': 2}, 'columns': {'train': ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'], 'validation': ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'], 'test': ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']}, 'batch_shapes': {'input_ids': (3, 6), 'attention_mask': (3, 6), 'labels': (3,), 'token_type_ids': (3, 6)}, 'batch_dtypes': {'input_ids': 'torch.int64', 'attention_mask': 'torch.int64', 'labels': 'torch.int64', 'token_type_ids': 'torch.int64'}, 'id_disjoint': True, 'mask_pad_consistent': True}
기본 문제 2 자동 검증: PASS


  **상세 해설** · `map`은 가변 길이 list를 Dataset에 저장하고, collator만 Tensor batch를 만듭니다. `id/text`는 오류 분석에 보존하지만 model collator에는 숫자 feature만 전달합니다.